# BQ2 shared helpers

**Goal:** one place for the calculations both BQ2 analysis tracks need: Track A (humidity and mould risk) and Track B (AC performance and degradation). If both notebooks use the same definitions, their zone tables can be merged into the risk matrix without mismatches.

**Inputs** (already in the repo):

| File | Used for |
|---|---|
| `data-cleaning/outputs/analytics/zone_hourly.csv` | Cleaned zone x hour table with `day_type` |
| `data-cleaning/outputs/clean/<year>Floor<f>.csv` | Cleaned minute data, for AC on/off events |
| `baseline/data/bangkok_era5_hourly_2018-2019.csv` | ERA5 weather cache, created by the baseline notebook |

**What this notebook defines:**

| Section | Contents |
|---|---|
| 1. Scope and data-quality flags | The 20 zones in scope and the known faulty sensor |
| 2. Psychrometrics | Dew point, humidity ratio, enthalpy, surface RH |
| 3. Seasons | Bangkok's hot, rainy and cool seasons |
| 4. Loading and merging | Zone table plus ERA5 weather and moisture removal |
| 5. AC on/off events | Start-up and shutdown detection from minute data |
| 6. Checks | Runs every helper on the real data to confirm it works |

**How to use it from another notebook** in this folder:

```python
RUN_CHECKS = False            # skip Section 6 so loading is instant
%run -i ./bq2_utils.ipynb     # every function and constant is now available
```

`%run` executes this notebook in your session, so the functions behave exactly as if they were defined in your own notebook. The `-i` flag lets this notebook see `RUN_CHECKS = False`. Without it, Section 6 runs every time, taking a few seconds.

**After editing this notebook,** re-run `%run -i ./bq2_utils.ipynb` in the other notebooks so they pick up the change, and tell your teammate, since it changes both sets of results.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

RUN_CHECKS = globals().get("RUN_CHECKS", True)   # set RUN_CHECKS = False before %run -i to skip Section 6

## 1. Scope and data-quality flags

**Scope:** Floors 3-7, zones 1, 2, 4 and 5, the 20 zones in the proposal.

- **Floor 1** has no temperature or humidity sensors.
- **Zone 3** on Floors 3-7 has no sensor either.
- **Floor 2** has a different layout (14 AC meters in one zone), so it wouldn't compare fairly.

**Known sensor problems that affect BQ2:**

- **Floor 5 zone 1:** the cleaning notebook (3.7, 6.5) found this sensor sits in the AC's cold supply air. Its temperature reads 15-17 °C in office hours. Cold air also reads *high* RH, so its humidity, dew point and mould metrics are biased too. It is kept in every table but flagged, and should be reported separately rather than ranked with the other 19 zones.
- **Sensor outage, 12 Sep 2018 to 6 Mar 2019:** no indoor sensor data on any floor. So 2018 vs 2019 like-for-like comparisons only work for Jul to mid-Sep, and the cool season (Nov-Feb) only has Nov-Dec 2019.

In [2]:
SCOPE_FLOORS = [3, 4, 5, 6, 7]
SCOPE_ZONES = [1, 2, 4, 5]

SUSPECT_SENSORS = {(5, 1): "Sensor in AC supply air: temperature biased cold, RH biased high (cleaning notebook 3.7/6.5)"}

SENSOR_OUTAGE = (pd.Timestamp("2018-09-12"), pd.Timestamp("2019-03-06"))

## 2. Psychrometrics

**Why not just use RH?** Relative humidity depends on temperature as well as moisture. The same air reads a higher RH when it is cooler, so a change in RH can mean the room got wetter *or* just got colder. These functions separate the two:

| Function | Measures | Why it matters for BQ2 |
|---|---|---|
| `dew_point_c` | Temperature at which water starts condensing (°C) | Any surface colder than this gets wet |
| `humidity_ratio_gkg` | Grams of water per kg of dry air | The actual amount of moisture, independent of temperature |
| `enthalpy_kjkg` | Total heat in the air, sensible + latent (kJ/kg) | How hard the AC has to work. In Bangkok much of the load is moisture, which temperature alone misses |
| `surface_rh_pct` | RH right next to a surface cooler than the room | Mould grows on surfaces (walls, diffusers, pipes), where RH is higher than in the room air |
| `sat_vp_hpa` | Saturation vapour pressure (hPa) | Building block for the others |

**Method:** Magnus formula with Sonntag (1990) coefficients, at standard sea-level pressure (1013.25 hPa). Bangkok is near sea level, so this is accurate enough.

In [3]:
def sat_vp_hpa(t_c):
    """Saturation vapour pressure (hPa), Magnus formula (Sonntag 1990 coefficients)."""
    return 6.112 * np.exp(17.62 * t_c / (243.12 + t_c))


def dew_point_c(t_c, rh_pct):
    """Dew point (°C) from air temperature (°C) and relative humidity (%)."""
    rh = np.clip(rh_pct, 1, 100)
    g = np.log(rh / 100) + 17.62 * t_c / (243.12 + t_c)
    return 243.12 * g / (17.62 - g)


def humidity_ratio_gkg(t_c, rh_pct, p_hpa=1013.25):
    """Grams of water vapour per kg of dry air."""
    pv = np.clip(rh_pct, 0, 100) / 100 * sat_vp_hpa(t_c)
    return 622 * pv / (p_hpa - pv)


def enthalpy_kjkg(t_c, w_gkg):
    """Moist-air enthalpy (kJ per kg dry air). Captures sensible + latent load in one number."""
    w = w_gkg / 1000
    return 1.006 * t_c + w * (2501 + 1.86 * t_c)


def surface_rh_pct(dew_c, surface_t_c):
    """RH at a surface colder than the room air (same moisture content, lower temperature)."""
    return np.clip(100 * sat_vp_hpa(dew_c) / sat_vp_hpa(surface_t_c), 0, 100)

## 3. Seasons

Bangkok's seasons (Thai Meteorological Department) are used instead of calendar months, because weather, and therefore AC load and indoor humidity, changes by season:

| Season | Months |
|---|---|
| Hot | Mar-May |
| Rainy | Jun-Oct |
| Cool | Nov-Feb |

The result is an ordered category, so tables and charts always list the seasons in this order.

In [4]:
SEASON_ORDER = ["hot", "rainy", "cool"]


def season(month):
    """Hot: Mar-May, rainy: Jun-Oct, cool: Nov-Feb. Works on an int or a Series of months."""
    m = pd.Series(month) if not isinstance(month, pd.Series) else month
    out = pd.Series(np.select([m.between(3, 5), m.between(6, 10)], ["hot", "rainy"], "cool"), index=m.index)
    return pd.Categorical(out, categories=SEASON_ORDER, ordered=True)

## 4. Loading and merging

- **`load_zone_hourly(path)`** reads the cleaned `zone_hourly.csv`, keeps the 20 scope zones, and adds `indoor_dew_C`, `indoor_w_gkg`, `season`, `year` and the `suspect_sensor` flag.
- **`add_outdoor_psychrometrics(weather)`** adds outdoor humidity ratio and enthalpy to the ERA5 table from the baseline's `load_bangkok_era5_weather()`.
- **`merge_weather(zone_hourly, weather)`** joins the weather onto the zone table by hour and adds **moisture removal** (`moisture_removal_gkg` = outdoor minus indoor humidity ratio).

**Why moisture removal matters:** it is how much drier the room is than outside. While the AC runs, that gap comes mostly from the AC condensing water out of the air. If the gap shrinks over time in similar weather, the AC is removing less moisture than it used to, a key sign of degradation that also raises mould risk.

In [5]:
def load_zone_hourly(path, scope_only=True):
    """Cleaned zone x hour table + indoor dew point, humidity ratio, season and quality flag."""
    zh = pd.read_csv(path, parse_dates=["Date"])
    if scope_only:
        zh = zh[zh["floor"].isin(SCOPE_FLOORS) & zh["zone"].astype(int).isin(SCOPE_ZONES)].copy()
    zh["zone"] = zh["zone"].astype(int)
    zh["indoor_dew_C"] = dew_point_c(zh["temp_C"], zh["rh_pct"])
    zh["indoor_w_gkg"] = humidity_ratio_gkg(zh["temp_C"], zh["rh_pct"])
    zh["season"] = season(zh["Date"].dt.month)
    zh["year"] = zh["Date"].dt.year
    zh["suspect_sensor"] = [(f, z) in SUSPECT_SENSORS for f, z in zip(zh["floor"], zh["zone"])]
    return zh


def add_outdoor_psychrometrics(weather):
    """Adds outdoor humidity ratio and enthalpy to the ERA5 table from load_bangkok_era5_weather()."""
    w = weather.copy()
    w["outdoor_w_gkg"] = humidity_ratio_gkg(w["outdoor_temp_C"], w["outdoor_rh_pct"])
    w["outdoor_enthalpy_kJkg"] = enthalpy_kjkg(w["outdoor_temp_C"], w["outdoor_w_gkg"])
    return w


def merge_weather(zone_hourly, weather):
    """Joins hourly ERA5 onto the zone table by timestamp and adds moisture removal (out - in)."""
    w = add_outdoor_psychrometrics(weather)
    w = w.reset_index().rename(columns={w.index.name or "index": "Date"})
    df = zone_hourly.merge(w, on="Date", how="left")
    df["moisture_removal_gkg"] = df["outdoor_w_gkg"] - df["indoor_w_gkg"]
    return df

## 5. AC on/off events

The AC switches off at lunch and in the evening, and back on in the morning and after lunch. Each switch is a natural experiment:

- **After a start-up**, how fast the room cools and dries shows the AC's capacity (**pull-down rate**, Track B).
- **After a shutdown**, how fast humidity climbs back shows how much moisture gets in (**rebound rate**, Track A).

These need **minute data**, because hourly averages blur the first 30 minutes after a switch.

**How an event is detected:**

1. **On or off:** the AC counts as "on" when its power is above 20% of the zone's 95th-percentile power, and at least 0.5 kW. The 0.5 kW floor keeps standby power (about 0.01-0.3 kW in the cleaning notebook) counted as "off".
2. **No flickers:** a switch only counts if the AC was in the old state for at least 30 minutes before it, and stays in the new state for at least 30 minutes after.
3. **No gaps:** the minute just before the switch must have a reading, so a data outage can't be mistaken for a shutdown.

`zone_minute_series(clean_dir, floor, zone)` loads one zone's minute data for both years: AC power summed over all its meters (only when every meter reports), temperature, RH and humidity ratio.

In [6]:
def ac_on_threshold(ac_kw):
    """'On' = above 20% of the zone's 95th-percentile AC power, and at least 0.5 kW (above standby)."""
    return max(0.5, 0.2 * ac_kw.quantile(0.95))


def detect_ac_events(ac_kw, min_state_minutes=30):
    """Start-ups and shutdowns from a minute-level AC kW series (DatetimeIndex).

    An event is kept only if the AC was in the old state for >= min_state_minutes before it and
    stays in the new state for >= min_state_minutes after it, so brief cycling isn't counted.
    Returns a DataFrame: time, event ('start' or 'stop'), threshold_kW.
    """
    ac_kw = ac_kw.dropna()
    thr = ac_on_threshold(ac_kw)
    on = (ac_kw > thr).astype(int)
    run_id = on.ne(on.shift()).cumsum()
    runs = pd.DataFrame({"state": on.groupby(run_id).first(),
                         "start": ac_kw.index.to_series().groupby(run_id).first(),
                         "minutes": on.groupby(run_id).size()}).reset_index(drop=True)
    events = []
    for i in range(1, len(runs)):
        prev, cur = runs.iloc[i - 1], runs.iloc[i]
        if prev.minutes < min_state_minutes or cur.minutes < min_state_minutes:
            continue
        before = cur.start - pd.Timedelta(minutes=1)
        if before not in ac_kw.index:      # a data gap, not a real switch
            continue
        events.append({"time": cur.start, "event": "start" if cur.state == 1 else "stop"})
    out = pd.DataFrame(events, columns=["time", "event"])
    out["threshold_kW"] = thr
    return out


def zone_minute_series(clean_dir, floor, zone):
    """Minute-level AC kW (all AC meters summed), temperature and RH for one zone, both years."""
    parts = [pd.read_csv(Path(clean_dir) / f"{y}Floor{floor}.csv", parse_dates=["Date"], index_col="Date")
             for y in (2018, 2019)]
    df = pd.concat(parts)
    ac_cols = [c for c in df.columns if c.startswith(f"z{zone}_AC")]
    out = pd.DataFrame(index=df.index)
    out["ac_kW"] = df[ac_cols].sum(axis=1, min_count=len(ac_cols)) if ac_cols else np.nan
    out["temp_C"] = df.get(f"z{zone}_S1(degC)")
    out["rh_pct"] = df.get(f"z{zone}_S1(RH%)")
    out["indoor_w_gkg"] = humidity_ratio_gkg(out["temp_C"], out["rh_pct"])
    return out

## 6. Checks

**What we're checking:** does every helper give sensible results on the real data? This section is skipped when another notebook loads this one with `RUN_CHECKS = False`.

### 6.1 Formulas against known values

At 24 °C and 60% RH, a standard psychrometric chart gives a dew point of about 15.8 °C and a humidity ratio of about 11.2 g/kg. Outdoor air at 30 °C and 20 g/kg has an enthalpy of about 81 kJ/kg.

In [7]:
if RUN_CHECKS:
    dew = dew_point_c(24, 60)
    display(pd.DataFrame([
        {"check": "Dew point, 24 °C / 60% RH", "result": round(dew, 2), "expected": "about 15.8 °C"},
        {"check": "Humidity ratio, 24 °C / 60% RH", "result": round(humidity_ratio_gkg(24, 60), 2), "expected": "about 11.2 g/kg"},
        {"check": "Enthalpy, 30 °C / 20 g/kg", "result": round(enthalpy_kjkg(30, 20), 1), "expected": "about 81 kJ/kg"},
        {"check": "Surface RH at 20 °C (room 24 °C / 60%)", "result": round(surface_rh_pct(dew, 20), 1), "expected": "about 77%, higher than the room"},
        {"check": "Seasons for Jan, Apr, Jul, Nov", "result": list(season(pd.Series([1, 4, 7, 11]))), "expected": "cool, hot, rainy, cool"},
    ]))

,check,result,expected
0,"Dew point, 24 °C / 60% RH",15.75,about 15.8 °C
1,"Humidity ratio, 24 °C / 60% RH",11.16,about 11.2 g/kg
2,"Enthalpy, 30 °C / 20 g/kg",81.3,about 81 kJ/kg
3,Surface RH at 20 °C (room 24 °C / 60%),76.6,"about 77%, higher than the room"
4,"Seasons for Jan, Apr, Jul, Nov","[cool, hot, rainy, cool]","cool, hot, rainy, cool"


**What this shows:** every value matches the psychrometric chart. A wall 4 °C colder than the room already sits at about 77% RH while the room reads 60%. That gap is why surface RH separates zones better than room RH.

### 6.2 Loading the zone table

In [8]:
if RUN_CHECKS:
    repo_root = next(p for p in [Path.cwd(), *Path.cwd().parents]
                     if (p / "data-cleaning").is_dir() and (p / "baseline").is_dir())
    CLEAN_DIR = repo_root / "data-cleaning" / "outputs" / "clean"
    ANALYTICS_DIR = repo_root / "data-cleaning" / "outputs" / "analytics"
    BASELINE_DIR = repo_root / "baseline"

    zh = load_zone_hourly(ANALYTICS_DIR / "zone_hourly.csv")
    print(f"{len(zh):,} zone-hours across {zh.groupby(['floor', 'zone']).ngroups} zones; "
          f"{zh['suspect_sensor'].groupby([zh['floor'], zh['zone']]).first().sum()} zone flagged as suspect")
    display(zh.groupby("season", observed=True).agg(
        zone_hours=("rh_pct", "size"),
        rh_coverage_pct=("rh_pct", lambda s: round(s.notna().mean() * 100, 1)),
        mean_indoor_dew_C=("indoor_dew_C", "mean"),
        mean_indoor_w_gkg=("indoor_w_gkg", "mean"),
    ).round(2))

263,520 zone-hours across 20 zones; 1 zone flagged as suspect


,zone_hours,rh_coverage_pct,mean_indoor_dew_C,mean_indoor_w_gkg
season,,,,
hot,44160,85.8,18.80,13.87
rainy,132480,75.0,18.36,13.40
cool,86880,25.2,16.01,11.49


**What this shows:** all 20 scope zones load, and only Floor 5 zone 1 is flagged. The hot and rainy seasons are well covered, but the **cool season has only about a quarter of its hours with humidity readings**, because of the Sep 2018 to Mar 2019 sensor outage. Seasonal findings for the cool season should be stated with that caveat.

### 6.3 AC events on one zone (Floor 3, zone 1)

In [9]:
if RUN_CHECKS:
    f3z1 = zone_minute_series(CLEAN_DIR, 3, 1)
    events = detect_ac_events(f3z1["ac_kW"])
    print(f"Threshold: {events['threshold_kW'].iloc[0]:.2f} kW; "
          f"{(events['event'] == 'start').sum()} starts, {(events['event'] == 'stop').sum()} stops")
    display(pd.crosstab(events["time"].dt.hour.rename("hour of day"), events["event"]))

Threshold: 5.75 kW; 626 starts, 583 stops


event,start,stop
hour of day,,
7,313,0
8,1,2
9,1,1
10,1,0
11,0,293
12,1,0
13,306,0
15,1,0
16,0,285


**What this shows:** the events cluster at the times the building's schedule predicts: start-ups at about 07:00 and 13:00, shutdowns before lunch and late afternoon. Only a handful fall at other hours. So the detector picks up real schedule switches rather than noise, and these are the events to measure pull-down and rebound from.

### 6.4 Weather merge

This uses the ERA5 cache that the baseline notebook downloads. If the cache doesn't exist yet, run `baseline/humidity_risk_baseline.ipynb` once (it needs internet the first time).

In [10]:
if RUN_CHECKS:
    cache = BASELINE_DIR / "data" / "bangkok_era5_hourly_2018-2019.csv"
    if cache.is_file():
        import sys
        sys.path.insert(0, str(BASELINE_DIR))
        from humidity_risk_baseline import load_bangkok_era5_weather
        df = merge_weather(zh, load_bangkok_era5_weather(cache))
        print(f"Rows with weather: {df['outdoor_temp_C'].notna().mean():.1%}")
        display(df.groupby("season", observed=True)[
            ["outdoor_w_gkg", "indoor_w_gkg", "moisture_removal_gkg", "outdoor_enthalpy_kJkg"]].mean().round(2))
    else:
        print(f"ERA5 cache not found at {cache.relative_to(repo_root).as_posix()}. "
              "Run baseline/humidity_risk_baseline.ipynb first.")

ERA5 cache not found at baseline/data/bangkok_era5_hourly_2018-2019.csv. Run baseline/humidity_risk_baseline.ipynb first.


**What to expect:** every row gets weather (ERA5 covers the whole period hourly). Indoor air should be several g/kg drier than outdoor air, so `moisture_removal_gkg` is positive on average. The analysis notebooks track how that gap changes per zone over time.